# Aggregate model

This is the simplest DES model of our system, which can be run for the whole trust or per county - just change the CSV file of parameters provided.

## Imports

In [1]:
import pandas as pd
import plotly.express as px
from rich import print  # noqa: A004

from ambdes import (
    ArrivalConfig,
    Model,
    ModelConfig,
    Results,
    Runner,
    SimConfig,
    TimesConfig,
    plot_warm_up,
    run_warm_up_audit,
)

## Data sources

All parameters for the model come from CSV files. These currently contain synthetic parameters. There are three files:

In [2]:
pd.read_csv("../data/param_arrivals.csv")

,Unnamed: 0,C1,C2,C3,C4
0,monday,25,310,180,40
1,tuesday,24,295,170,38
2,wednesday,24,295,170,38
3,thursday,24,295,170,38
4,friday,25,305,178,40
5,saturday,28,340,200,45
6,sunday,27,330,195,43


In [3]:
pd.read_csv("../data/param_times.csv")

,time,type,C1,C2,C3,C4
0,mobilisation,mean,3.0,3.0,3.0,3.0
1,mobilisation,sd,0.5,0.5,0.5,0.5
2,travel_to_scene,mean,8.0,10.0,12.0,12.0
3,travel_to_scene,sd,5.0,5.0,5.0,5.0
4,on_scene,mean,44.0,46.0,48.0,50.0
5,on_scene,sd,5.0,5.0,5.0,5.0
6,travel_to_hospital,mean,8.0,10.0,12.0,12.0
7,travel_to_hospital,sd,5.0,5.0,5.0,5.0
8,handover,mean,15.0,22.0,40.0,45.0
9,handover,sd,5.0,5.0,5.0,5.0


In [4]:
pd.read_csv("../data/param_model.csv")

,parameter,value
0,resource_hours_per_week,52000
1,warm_up_period,500
2,data_collection_period,10080
3,n_reps,5


## Walk through the model

### Arrivals

The `ArrivalConfig` class loads `arrivals.csv` and uses it to define:

* **Patient arrivals:** a non-homogeneous Poisson process (NHPP) that varies by day of week. With a Poisson distribution, inter-arrival times are exponentially distributed.
* **Response categories:** assigned probabilistically based on observed proportions of each category.

In [5]:
arrival_config = ArrivalConfig(arrival_csv="../data/param_arrivals.csv")

# Parameters used in NHPP
display(arrival_config.nspp_df)

display(arrival_config.category_proportions)

,t,mean_iat
0,0,2.594595
1,1440,2.732448
2,2880,2.732448
3,4320,2.732448
4,5760,2.627737
5,7200,2.349103
6,8640,2.420168


C1    0.045478
C2    0.557674
C3    0.324411
C4    0.072437
dtype: float64

Our assumptions for arrivals are that:

* (a) Inter-arrival times vary by response category and day of week.
    * We agreed this in our discussion.
* (b) Proportion of each response category *do not* vary by day of week.
    * To check this assumption, we need to look at the proportion of C1 v.s., C2 v.s., C3 v.s., C4 by day of week...

In [6]:
# Checking proportion of each response category by day of week
display(arrival_config.proportion_df)
display(arrival_config.variation_df)

,C1,C2,C3,C4
monday,0.045045,0.558559,0.324324,0.072072
tuesday,0.045541,0.559772,0.322581,0.072106
wednesday,0.045541,0.559772,0.322581,0.072106
thursday,0.045541,0.559772,0.322581,0.072106
friday,0.045620,0.556569,0.324818,0.072993
saturday,0.045677,0.554649,0.326264,0.073409
sunday,0.045378,0.554622,0.327731,0.072269


,mean,min,max,range,sd
C1,0.045478,0.045045,0.045677,0.000632,0.000212
C2,0.557674,0.554622,0.559772,0.005150,0.002369
C3,0.324411,0.322581,0.327731,0.005150,0.002028
C4,0.072437,0.072072,0.073409,0.001337,0.000539


### Times

The `TimesConfig` class loads `times.csv`. Currently, times are all modelled as lognormal distributions, so a mean and standard deviation (SD) is provided for each.

We have assumed that each of these times *vary by response category* - though we should check this in the real data - the model could be simplified to just one time across categories if it doesn't vary.

In [7]:
times_config = TimesConfig(times_csv="../data/param_times.csv")
display(times_config.times_df)

,time,type,C1,C2,C3,C4
0,mobilisation,mean,3.0,3.0,3.0,3.0
1,mobilisation,sd,0.5,0.5,0.5,0.5
2,travel_to_scene,mean,8.0,10.0,12.0,12.0
3,travel_to_scene,sd,5.0,5.0,5.0,5.0
4,on_scene,mean,44.0,46.0,48.0,50.0
5,on_scene,sd,5.0,5.0,5.0,5.0
6,travel_to_hospital,mean,8.0,10.0,12.0,12.0
7,travel_to_hospital,sd,5.0,5.0,5.0,5.0
8,handover,mean,15.0,22.0,40.0,45.0
9,handover,sd,5.0,5.0,5.0,5.0


## Other parameters

The other parameters are imported by `ModelConfig`.

In [8]:
model_config = ModelConfig(param_csv="../data/param_model.csv")

### Configuration

The `SimConfig` class accepts instances of the config classes above, and uses them to create a single set of parameters ready for the model.

The distributions are all stored in `dist_config`, which follows a JSON format that can be accepted by the `sim-tools` `DistributionRegistry` class.

In [9]:
config = SimConfig(
    arrival_config=arrival_config,
    times_config=times_config,
    model_config=model_config,
)
print(config.__dict__)

{
    'dist_config': {
        'call_arrival': {
            'class_name': 'NSPPThinning',
            'params': {
                'data':       t  mean_iat
0     0  2.594595
1  1440  2.732448
2  2880  2.732448
3  4320  2.732448
4  5760  2.627737
5  7200  2.349103
6  8640  2.420168
            }
        },
        'call_category': {
            'class_name': 'DiscreteEmpirical',
            'params': {
                'values': Index(['C1', 'C2', 'C3', 'C4'], dtype='str'),
                'freq': array([0.04547757, 0.5576737 , 0.32441131, 0.07243742])
            }
        },
        'mobilisation_time': {
            'C1': {'class_name': 'Lognormal', 'params': {'mean': 3.0, 'stdev': 0.5}},
            'C2': {'class_name': 'Lognormal', 'params': {'mean': 3.0, 'stdev': 0.5}},
            'C3': {'class_name': 'Lognormal', 'params': {'mean': 3.0, 'stdev': 0.5}},
            'C4': {'class_name': 'Lognormal', 'params': {'mean': 3.0, 'stdev': 0.5}}
        },
        'time_to_scene': {
            'C1': {'class_name': 'Lognormal', 'params': {'mean': 8.0, 'stdev': 5.0}},
            'C2': {'class_name': 'Lognormal', 'params': {'mean': 10.0, 'stdev': 5.0}},
            'C3': {'class_name': 'Lognormal', 'params': {'mean': 12.0, 'stdev': 5.0}},
            'C4': {'class_name': 'Lognormal', 'params': {'mean': 12.0, 'stdev': 5.0}}
        },
        'on_scene_time': {
            'C1': {'class_name': 'Lognormal', 'params': {'mean': 44.0, 'stdev': 5.0}},
            'C2': {'class_name': 'Lognormal', 'params': {'mean': 46.0, 'stdev': 5.0}},
            'C3': {'class_name': 'Lognormal', 'params': {'mean': 48.0, 'stdev': 5.0}},
            'C4': {'class_name': 'Lognormal', 'params': {'mean': 50.0, 'stdev': 5.0}}
        },
        'time_to_hospital': {
            'C1': {'class_name': 'Lognormal', 'params': {'mean': 8.0, 'stdev': 5.0}},
            'C2': {'class_name': 'Lognormal', 'params': {'mean': 10.0, 'stdev': 5.0}},
            'C3': {'class_name': 'Lognormal', 'params': {'mean': 12.0, 'stdev': 5.0}},
            'C4': {'class_name': 'Lognormal', 'params': {'mean': 12.0, 'stdev': 5.0}}
        },
        'handover_time': {
            'C1': {'class_name': 'Lognormal', 'params': {'mean': 15.0, 'stdev': 5.0}},
            'C2': {'class_name': 'Lognormal', 'params': {'mean': 22.0, 'stdev': 5.0}},
            'C3': {'class_name': 'Lognormal', 'params': {'mean': 40.0, 'stdev': 5.0}},
            'C4': {'class_name': 'Lognormal', 'params': {'mean': 45.0, 'stdev': 5.0}}
        },
        'wrap_up_time': {
            'C1': {'class_name': 'Lognormal', 'params': {'mean': 5.0, 'stdev': 2.0}},
            'C2': {'class_name': 'Lognormal', 'params': {'mean': 5.0, 'stdev': 2.0}},
            'C3': {'class_name': 'Lognormal', 'params': {'mean': 5.0, 'stdev': 2.0}},
            'C4': {'class_name': 'Lognormal', 'params': {'mean': 5.0, 'stdev': 2.0}}
        }
    },
    'n_ambulances': 310,
    'warm_up_period': 500,
    'data_collection_period': 10080,
    'n_reps': 5
}

### Model

The `Model` can be set-up by setting a run number and providing the `config` instance, then run by calling `run()`.

In [10]:
model = Model(run_number=0, config=config)
model.run()

### Logger

We record a log using the `vidigi` `EventLogger` class. This means it can work with `vidigi` to produce animations or process flow charts if desired.

We also have the attributes each patient stored in the model - for example, here, we can look at the patient with ID one in `model.patients` and in the `log`.

In [11]:
log = model.logger.to_dataframe()

# View patient with ID 1
print(model.patients[0].__dict__)
display(log[log["entity_id"] == 1])

{'patient_id': 1, 'category': 'C2', 'call_timestamp': 501.8479742056042, 'response_time': 16.768964975497113}

,entity_id,event_type,event,time,run_number,resource_id
0,1,arrival_departure,arrival,7.736835,0,NaN
1,1,queue,ambulance_wait_begins,7.736835,0,NaN
2,1,resource_use,ambulance_assigned,7.736835,0,1.0
114,1,resource_use_end,ambulance_available,122.449830,0,1.0
115,1,arrival_departure,depart,122.449830,0,NaN
844,1,arrival_departure,arrival,501.847974,0,NaN
845,1,queue,ambulance_wait_begins,501.847974,0,NaN
846,1,resource_use,ambulance_assigned,501.847974,0,189.0
1104,1,resource_use_end,ambulance_available,610.374821,0,189.0
1105,1,arrival_departure,depart,610.374821,0,NaN


## Results

The `Results` class accepts the run model instance and can calculate various performance measures.

In [12]:
Results(model).summary_df()

,category,n_patients,mean_response_time,run,mean_utilisation
0,C1,177.0,11.018527,0,NaN
1,C2,2177.0,13.036596,0,NaN
2,C3,1233.0,15.070640,0,NaN
3,C4,302.0,14.785669,0,NaN
4,all,NaN,NaN,0,0.150795


In [13]:
Results(model).utilisation_df()

,time,busy,interval_duration,utilisation
0,500.000000,238,1.007559,0.767742
1,501.007559,237,0.008792,0.764516
2,501.016351,236,0.831623,0.761290
3,501.847974,237,0.514070,0.764516
4,502.362044,238,0.807198,0.767742
...,...,...,...,...
7788,10573.492283,34,0.624036,0.109677
7789,10574.116319,35,0.208256,0.112903
7790,10574.324574,36,3.338483,0.116129
7791,10577.663057,37,0.054213,0.119355


### Runner

A `Runner` class is provided to run the model once or for multiple replications.

It also uses the `Results` class to calculate some performance measures.

In [14]:
runner = Runner(config)
results = runner.run_reps()

## Results

This is an example of running the model for one replication, for one week, with no warm-up period.

### Mean response time and utilisation

We can view by run and overall.

In [15]:
display(results["run"])

,category,n_patients,mean_response_time,run,mean_utilisation
0,C1,177.0,11.018527,0,NaN
1,C2,2177.0,13.036596,0,NaN
2,C3,1233.0,15.070640,0,NaN
3,C4,302.0,14.785669,0,NaN
4,all,NaN,NaN,0,0.150795
5,C1,164.0,10.778153,1,NaN
6,C2,2085.0,12.938092,1,NaN
7,C3,1384.0,15.107567,1,NaN
8,C4,277.0,14.960569,1,NaN
9,all,NaN,NaN,1,0.153510


In [16]:
display(results["overall"])

,category,mean_n_patients,mean_response_time,mean_utilisation
0,C1,174.6,10.933270,NaN
1,C2,2167.8,12.936119,NaN
2,C3,1275.2,15.050279,NaN
3,C4,284.2,14.983728,NaN
4,all,NaN,NaN,0.153704


### Example plot: distribution of response times by response category

This is an example taking results from a single model instance, and plotting distribution of response times observed during that run.

In [17]:
df = pd.DataFrame(
    {
        "response_time": [p.response_time for p in model.patients],
        "category": [p.category for p in model.patients],
    }
)

fig = px.histogram(
    df,
    x="response_time",
    facet_col="category",
    category_orders={"category": ["C1", "C2", "C3", "C4"]},
    labels={
        "response_time": "Response time (minutes)",
        "category": "Category",
    },
    title="Distribution of response times by category",
)

fig.layout.yaxis.title.text = "Number of patients"

fig.show()

## Determine appropriate warm-up period length

We can use the time series inspection approach to decide how long our warm-up period should be.

This requires recording performance measures at regular intervals. I have functions and classes to enable this analysis in `choose_warm_up.py`.

In [18]:
# Run audit
audit = run_warm_up_audit(config=config, interval=30, n_reps=5)

# Preview the dataframe produced
audit.head(20)

,time,category,metric,value,run
0,0,C1,response_time,NaN,0
1,0,C2,response_time,NaN,0
2,0,C3,response_time,NaN,0
3,0,C4,response_time,NaN,0
4,0,all,utilisation,0.000000,0
5,30,C1,response_time,NaN,0
6,30,C2,response_time,9.155393,0
7,30,C3,response_time,15.262418,0
8,30,C4,response_time,NaN,0
9,30,all,utilisation,0.012986,0


### Example: C1 mean response time

In [19]:
plot_warm_up(audit=audit, metric="response_time", category="C1")

### Example: Mean utilisation

In [20]:
plot_warm_up(audit=audit, metric="utilisation")